# 12 -- ORCA: Capabilities & Practical Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/12_orca_tutorial.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:
- Write valid ORCA input files for single-point, optimization, and frequency jobs
- Understand the `!` keyword line and `%` block syntax
- Use the RIJCOSX approximation to accelerate hybrid-DFT calculations
- Set up DLPNO-CCSD(T) for benchmark-quality energetics
- Parse ORCA output files with Python and cclib
- Choose appropriate methods, basis sets, and auxiliary basis sets
- Recognize common ORCA error messages and their solutions

## 1. ORCA Overview

**ORCA** (developed by Frank Neese, MPI Mulheim) is a free-for-academic production
quantum chemistry code. Download from https://orcaforum.kofo.mpg.de

| Area | Methods |
|------|--------|
| DFT | All major XC functionals; hybrid, double-hybrid |
| Post-HF | MP2, CCSD(T), DLPNO-CCSD(T) |
| Multireference | CASSCF, NEVPT2, MRCI |
| Spectroscopy | EPR, Mossbauer, X-ray, NMR |
| Relativistic | ZORA, DKH, CPCM solvation |

## 2. Input File Syntax

An ORCA input file (`job.inp`) consists of:
1. **`!` keyword line** -- method, basis set, and job-type keywords
2. **`%` blocks** -- detailed options (optional)
3. **`* xyz` coordinate block** -- molecular geometry

### 2.1 Minimal Single-Point

```
! B3LYP def2-SVP TightSCF

* xyz 0 1
  O   0.000   0.000   0.117
  H   0.000   0.757  -0.469
  H   0.000  -0.757  -0.469
*
```

### 2.2 Geometry Optimization + Frequencies

```
! B3LYP def2-TZVP TightSCF Opt Freq

%maxcore 4000
%pal nprocs 8 end

* xyz 0 1
  O   0.000   0.000   0.117
  H   0.000   0.757  -0.469
  H   0.000  -0.757  -0.469
*
```

### 2.3 RIJCOSX for Fast Hybrid DFT

The **RIJCOSX** approximation (RI-J for Coulomb + COSX for exchange) reduces
the cost of hybrid functionals from O(N^4) to ~O(N^3):

```
! B3LYP def2-TZVP def2/J RIJCOSX TightSCF Opt
```

Always pair with a matching auxiliary basis: `def2/J` for Coulomb fitting;
`def2-TZVP/C` for correlation (MP2/CC).

## 3. DLPNO-CCSD(T): Gold-Standard Energetics

**DLPNO-CCSD(T)** (Domain-based Local Pair Natural Orbital) achieves near-canonical
CCSD(T) accuracy at a fraction of the cost (O(N^3) vs O(N^7)):

```
! DLPNO-CCSD(T) def2-TZVP def2-TZVP/C TightSCF TightPNO

%maxcore 8000
%pal nprocs 16 end

* xyz 0 1
  ...
*
```

**Typical accuracy**: Delta-H errors < 1 kcal/mol vs canonical CCSD(T).

**TightPNO thresholds** (recommended for benchmarks):
- `TCutPNO = 1e-7` (default Normal = 3.33e-7)
- `TCutPairs = 1e-5`
- `TCutMKN = 1e-3`

In [ ]:
# =============================================================================
# Ch121a: Quantum Chemistry & DFT -- Notebook 12: ORCA Tutorial
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================

# ------------------------------------------------------------------
# Writing ORCA Input Files with Python
# ------------------------------------------------------------------

def write_orca_input(filename, method, basis, charge, mult, atoms,
                     extra_keywords='', extra_blocks=''):
    '''Write a minimal ORCA .inp file.'''
    coord_block = '\n'.join(
        f'  {sym:2s}  {x:10.5f}  {y:10.5f}  {z:10.5f}'
        for sym, x, y, z in atoms
    )
    content = f'! {method} {basis} TightSCF {extra_keywords}\n'
    if extra_blocks:
        content += extra_blocks + '\n'
    content += f'\n* xyz {charge} {mult}\n{coord_block}\n*\n'
    with open(filename, 'w') as f:
        f.write(content)
    return content

water_atoms = [
    ('O',  0.000,  0.000,  0.117),
    ('H',  0.000,  0.757, -0.469),
    ('H',  0.000, -0.757, -0.469),
]

inp_sp = write_orca_input(
    '/tmp/water_sp.inp',
    method='B3LYP', basis='def2-TZVP',
    charge=0, mult=1, atoms=water_atoms,
    extra_keywords='def2/J RIJCOSX'
)
print('=== Water Single-Point (B3LYP/def2-TZVP/RIJCOSX) ===')
print(inp_sp)

# Example: Fe(CO)5 optimization
feco5_atoms = [
    ('Fe',  0.000,  0.000,  0.000),
    ('C',   0.000,  0.000,  1.807), ('C',  0.000,  0.000, -1.807),
    ('C',   1.807,  0.000,  0.000), ('C', -1.807,  0.000,  0.000),
    ('C',   0.000,  1.807,  0.000),
    ('O',   0.000,  0.000,  2.975), ('O',  0.000,  0.000, -2.975),
    ('O',   2.975,  0.000,  0.000), ('O', -2.975,  0.000,  0.000),
    ('O',   0.000,  2.975,  0.000),
]
inp_opt = write_orca_input(
    '/tmp/feco5_opt.inp',
    method='BP86', basis='def2-TZVP',
    charge=0, mult=1, atoms=feco5_atoms,
    extra_keywords='def2/J RI Opt',
    extra_blocks='%maxcore 4000\n%pal nprocs 4 end'
)
print('=== Fe(CO)5 Optimization (BP86/def2-TZVP/RI) ===')
print(inp_opt)

In [ ]:
# ------------------------------------------------------------------
# Parsing ORCA Output Files
# ------------------------------------------------------------------
import re

mock_orca_out = '''
ORCA SCF CONVERGED AFTER  11 CYCLES
Total Energy       :         -76.437879956 Eh
TOTAL ENERGY          -76.437879956 Hartree
TOTAL ENTHALPY        -76.435432876 Hartree

Mulliken charges:
   0 O   :  -0.708
   1 H   :   0.354
   2 H   :   0.354

VIBRATIONAL FREQUENCIES
   0:       0.00 cm**-1
   3:    1641.82 cm**-1
   4:    3779.56 cm**-1
   5:    3892.41 cm**-1
'''

e_match = re.search(r'TOTAL ENERGY\s+([\-0-9.]+)\s+Hartree', mock_orca_out)
charge_matches = re.findall(r'\s+\d+\s+(\w+)\s+:\s+([\-0-9.]+)', mock_orca_out)
freq_matches = re.findall(r'\s+\d+:\s+([0-9.]+) cm\*\*-1', mock_orca_out)

print('Parsed from mock ORCA .out:')
if e_match:
    print(f'  Total energy = {e_match.group(1)} Hartree')
print('  Mulliken charges:', {sym: float(q) for sym, q in charge_matches})
non_zero = [f for f in freq_matches if float(f) > 10]
print('  Vibrational frequencies (cm-1):', non_zero)

print()
print('--- cclib usage (with a real ORCA .out file) ---')
print('import cclib')
print('data = cclib.io.ccread("water.out")')
print('print(data.scfenergies[-1], "eV")')
print('print(data.atomcharges["mulliken"])')
print('print(data.vibfreqs, "cm-1")')

## 4. Common ORCA Keywords Reference

| Keyword | Purpose |
|---------|--------|
| `TightSCF` / `VeryTightSCF` | SCF convergence criteria |
| `Opt` | Geometry optimization |
| `Freq` | Analytical or numerical Hessian |
| `NMR` | NMR shielding tensors |
| `RIJCOSX` | RI-J Coulomb + seminumerical exchange (hybrids) |
| `RI` / `RIJONX` | RI approximation for pure DFT |
| `CPCM(Water)` | Conductor-like PCM solvation |
| `def2/J` | Auxiliary Coulomb-fitting basis |
| `def2-TZVP/C` | Auxiliary correlation basis for MP2/CC |
| `TightPNO` | Tight thresholds for DLPNO methods |
| `ZORA` / `DKH` | Scalar relativistic corrections |
| `D3BJ` | Grimme D3 dispersion with Becke-Johnson damping |
| `Grid5` / `Grid7` | Integration grid accuracy |

## Research Connection

ORCA is used across computational chemistry research:

- **Bioinorganic chemistry**: DLPNO-CCSD(T) energetics for iron-sulfur clusters and
  metalloenzyme active sites too large for canonical CC.
- **Catalysis**: BP86/TZVP geometry optimizations followed by DLPNO-CCSD(T) single-points
  is a standard protocol for reaction energy benchmarks.
- **Spectroscopy**: ORCA's EFG, EPR, and Mossbauer modules interpret transition metal spectra.
- **ML potentials**: ORCA generates training data for neural network potentials
  (e.g., wB97X-D3/def2-TZVP for ANI-2x).

## Summary

| Topic | Key Point |
|-------|----------|
| Input syntax | `! method basis [keywords]` + `* xyz charge mult ... *` |
| RIJCOSX | RI-J Coulomb + COSX exchange; ~10x faster hybrids |
| DLPNO-CCSD(T) | O(N^3) near-canonical CCSD(T); < 1 kcal/mol error |
| Auxiliary basis | `def2/J` for RI-J; `def2-TZVP/C` for correlation |
| Output parsing | cclib: `cclib.io.ccread('file.out')` |
| Parallelism | `%pal nprocs N end`; `%maxcore MB_per_core` |
| Grid | `Grid5` default; `Grid7` for final energies |

## Exercises

1. **Write an input**: Write an ORCA input for geometry optimization of ethanol
   (CH3CH2OH) using B3LYP-D3BJ/def2-TZVP with RIJCOSX. Include `%pal` and
   `%maxcore` blocks for 8 cores and 4 GB/core.

2. **DLPNO setup**: Write a DLPNO-CCSD(T)/def2-TZVP single-point input for
   the optimized ethanol geometry. Use TightPNO settings.

3. **cclib parsing**: Given a real ORCA frequency output, write Python code using
   cclib to (a) extract all vibrational frequencies, (b) compute the ZPE, and
   (c) identify imaginary modes.

4. **Basis set convergence**: Design an ORCA input series using def2-SVP, def2-TZVP,
   and def2-QZVP for HF single-points on the HF molecule.

5. **RIJCOSX accuracy**: Explain the RIJCOSX approximation. What property calculations
   might be sensitive to the COSX approximation for the exchange part?